In [1]:
# ==============================================================================
# ROGII Wellbore Geology — GPU Sequence Model (1D U-Net over u = TVT + Z)
# ==============================================================================
# HOW TO USE: paste each numbered block below into its own Kaggle notebook cell.
# Requires: Kaggle notebook with GPU accelerator (Settings -> Accelerator -> GPU).
# torch is preinstalled on Kaggle GPU images. No pip install needed.
#
# DESIGN (why this can beat the 8.9 classical ceiling):
#   * Predicts u = TVT + Z, the SMOOTH structural curve (TVT inherits Z's high
#     frequency; u does not). Model outputs u_hat; final TVT = u_hat - Z.
#   * 1D U-Net sees the WHOLE well sequence, learning multi-scale curvature the
#     station-wise DP cannot represent.
#   * Known-zone true u is fed as an input channel (it is given); loss is masked
#     to the blind zone only.
#   * Self-excluded spatial field is an input channel — the model learns WHEN to
#     trust it vs the GR shape.
#   * Typewell GR-vs-u curve is summarized as global context features.
#
# CRITICAL LEAK GUARDS (verify these as you run — they are why results are real):
#   G1. Field features exclude each well's OWN lateral (deployment-identical).
#   G2. Cross-validation splits by WELL, never by station.
#   G3. Loss and all validation metrics are computed on BLIND stations only.
#   G4. Normalization stats (feature means/stds) are fit on TRAIN wells only.
# ==============================================================================


# ==============================================================================
# CELL 1 — Imports, config, device
# ==============================================================================
import os, glob, math, time, json, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial import cKDTree

torch.manual_seed(0); np.random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)
assert DEVICE == 'cuda', 'Enable GPU: Settings -> Accelerator -> GPU P100'

# Competition data root on Kaggle (train/ is present in the rerun; test/ swaps to hidden set)
# ROOT = '/kaggle/input/rogii-wellbore-geology-prediction'
ROOT = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'
TRAIN_DIR = os.path.join(ROOT, 'train')
TEST_DIR = os.path.join(ROOT, 'test')

CFG = dict(
    win_max=12288,       # was 4096 — covers max well length 12141 (p99 11031)
    field_sub=4,
    field_k=40, field_soft=400.0,
    sup_cut=900.0,
    n_folds=5,
    hidden=96,
    lr=1e-3, weight_decay=1e-4,
    epochs=250, patience=60, lr_min_frac=0.02, batch_wells=8,
    aug_gain=0.15, aug_offset=8.0,
    anchor_band=200,     # NEW (lever 1): known stations before boundary added to TRAIN loss
    ema_decay=0.999,     # NEW (lever 2): per-step weight EMA
    ema_warmup=20,       # NEW: don't evaluate EMA before this epoch (it's meaningless early)
    ema_eval_every=5,    # NEW: evaluate EMA every N epochs — cost control
)
print(CFG)

device: cuda | torch 2.10.0+cu128
{'win_max': 12288, 'field_sub': 4, 'field_k': 40, 'field_soft': 400.0, 'sup_cut': 900.0, 'n_folds': 5, 'hidden': 96, 'lr': 0.001, 'weight_decay': 0.0001, 'epochs': 250, 'patience': 60, 'lr_min_frac': 0.02, 'batch_wells': 8, 'aug_gain': 0.15, 'aug_offset': 8.0, 'anchor_band': 200, 'ema_decay': 0.999, 'ema_warmup': 20, 'ema_eval_every': 5}


In [2]:
# ==============================================================================
# CELL 2 — Load all wells into memory (train + the 3 example test wells)
# ==============================================================================
H_COLS = ['MD','X','Y','Z','TVT','GR','TVT_input']

def list_wells(d):
    return sorted(os.path.basename(f).split('__')[0]
                  for f in glob.glob(os.path.join(d, '*__horizontal_well.csv')))

def load_well(split_dir, w):
    h = pd.read_csv(os.path.join(split_dir, f'{w}__horizontal_well.csv'))
    t = pd.read_csv(os.path.join(split_dir, f'{w}__typewell.csv'))
    return h, t

train_wells = list_wells(TRAIN_DIR)
test_wells = list_wells(TEST_DIR)
print(f'train wells: {len(train_wells)} | test wells: {len(test_wells)}')

# Detect Z sign convention once (TVD below sea level is negative in this data).
# u = TVT + Z should be SMOOTH; if Z has the wrong sign, u is noisy. Fix per data.
def zfix(h):
    # Empirically Z is negative (below sea level). u = TVT + Z is the smooth curve.
    return h['Z'].values.astype(np.float64)

train wells: 773 | test wells: 3


In [3]:
# ==============================================================================
# CELL 3 — Build the spatial structural field, WID-tagged for self-exclusion (G1)
# ==============================================================================
# u datum: align each well by its typewell's primary formation top so u values are
# comparable across wells. We approximate the datum by the well's own known-zone
# median (u_known), which needs no Geology column and mirrors deployment.
#
# The field stores (x, y) -> u points from EVERY training lateral, tagged by well id.
# Querying with self_well=W excludes W's own points => leak-free (G1).

def build_field(wells, split_dir):
    pts, us, wid = [], [], []
    for w in wells:
        h, _ = load_well(split_dir, w)
        z = zfix(h)
        tvt = h['TVT'].values.astype(np.float64)
        x, y = h['X'].values, h['Y'].values
        m = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & np.isfinite(tvt)
        if m.sum() < 200:
            continue
        u = (tvt + z)[m]
        # per-well datum: center u on its own median so cross-well u is comparable
        u = u - np.median(u)
        s = slice(None, None, CFG['field_sub'])
        pts.append(np.column_stack([x[m][s], y[m][s]]))
        us.append(u[s]); wid += [w] * len(pts[-1])
    P = np.vstack(pts); U = np.concatenate(us); WID = np.array(wid)
    return cKDTree(P), U, WID

# NOTE: we also need each well's datum (median u) to map field u back to that well's
# frame. We store it as we go in build_field_with_datum below.
def build_field_with_datum(wells, split_dir):
    pts, us, wid = [], [], []
    datum = {}
    for w in wells:
        h, _ = load_well(split_dir, w)
        z = zfix(h); tvt = h['TVT'].values.astype(np.float64)
        x, y = h['X'].values, h['Y'].values
        m = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & np.isfinite(tvt)
        if m.sum() < 200:
            continue
        u_raw = (tvt + z)[m]
        d = float(np.median(u_raw))
        datum[w] = d
        s = slice(None, None, CFG['field_sub'])
        pts.append(np.column_stack([x[m][s], y[m][s]]))
        us.append((u_raw - d)[s]); wid += [w] * len(pts[-1])
    P = np.vstack(pts); U = np.concatenate(us); WID = np.array(wid)
    return cKDTree(P), U, WID, datum, P

print('building field over all training wells...')
t0 = time.time()
FIELD_TREE, FIELD_U, FIELD_WID, FIELD_DATUM, FIELD_POINTS = build_field_with_datum(train_wells, TRAIN_DIR)
print(f'field: {len(FIELD_U)} points, {len(set(FIELD_WID))} wells ({time.time()-t0:.0f}s)')

# Per-well cache of self-excluded trees. A well sits ON its own dense lateral, so
# masking neighbors by id after a shared-tree query removes ALL of them. The correct
# leak-free fix (G1) is to REBUILD the tree without the well's own points, then query.
_EXCL_CACHE = {}
def field_query(xq, yq, self_well):
    """Self-excluded field estimate of centered-u (G1): exclude-then-build."""
    if self_well not in _EXCL_CACHE:
        keep = FIELD_WID != self_well
        _EXCL_CACHE[self_well] = (cKDTree(FIELD_POINTS[keep]), FIELD_U[keep])
        # bound cache memory: keep at most ~24 recent trees
        if len(_EXCL_CACHE) > 24:
            _EXCL_CACHE.pop(next(iter(_EXCL_CACHE)))
    tree_ex, U_ex = _EXCL_CACHE[self_well]
    dd, idx = tree_ex.query(np.column_stack([xq, yq]), k=CFG['field_k'])
    wq = 1.0 / (dd + CFG['field_soft']) ** 2
    s = wq.sum(1)
    est = np.einsum('nk,nk->n', wq, U_ex[idx]) / np.maximum(s, 1e-12)
    dmin = dd[:, 0]
    est[s <= 1e-12] = np.nan
    return est, dmin


building field over all training wells...
field: 1273344 points, 773 wells (22s)


In [4]:
# ==============================================================================
# CELL 4 — Typewell global context (physical anchor, likely the key edge)
# ==============================================================================
# The typewell is GR vs TVT(=u index) in the vertical reference. We summarize it as
# a fixed-length vector by resampling GR onto a standard u-grid, so the model gets
# the vertical GR signature to correlate against.

TW_GRID = np.linspace(-400, 400, 64)   # standard u-offset grid (ft) around datum

def typewell_context(t, datum):
    tvt = t['TVT'].values.astype(np.float64)
    gr = t['GR'].values.astype(np.float64)
    m = np.isfinite(tvt) & np.isfinite(gr)
    if m.sum() < 5:
        return np.zeros(len(TW_GRID), np.float32)
    # center on this well's datum; typewell TVT already ~ u index
    u = tvt[m] - datum
    order = np.argsort(u)
    g = np.interp(TW_GRID, u[order], gr[m][order],
                  left=gr[m][order][0], right=gr[m][order][-1])
    g = (g - np.nanmedian(gr)) / 30.0
    return g.astype(np.float32)

In [5]:
# ==============================================================================
# CELL 5 — Per-well sequence tensor builder (leak-free features)
# ==============================================================================
# Channels (per station), all normalized to be well-invariant:
#   0 GR (median-centered /30)      1 GR smoothed        2 dGR
#   3 field_u (centered /20)        4 field support 0..1 5 known-mask (1 in known)
#   6 known u where known else 0    7 Z detrended /50    8 position through well
# Target: u (centered by datum). Loss masked to BLIND stations.

def build_sequence(h, t, self_well, datum, is_test=False):
    z = zfix(h)
    x, y = h['X'].values, h['Y'].values
    gr = h['GR'].values.astype(np.float64)
    tvt_in = h['TVT_input'].values.astype(np.float64)
    md = h['MD'].values.astype(np.float64)
    n = len(h)
    blind = ~np.isfinite(tvt_in)                 # blind zone = TVT_input is NaN
    med = np.nanmedian(gr)
    grf = np.where(np.isfinite(gr), gr, med)
    grs = np.convolve(grf, np.ones(15)/15, mode='same')
    dgr = np.gradient(grs)

    fe, db = field_query(x, y, self_well)        # self-excluded (G1)
    # interpolate field NaNs for continuity (center support still tracked in ch4)
    fm = np.isfinite(fe)
    if fm.sum() >= 2:
        fe = np.interp(np.arange(n), np.where(fm)[0], fe[fm])
    else:
        fe = np.zeros(n)

    known = ~blind
    # reference u at the known/blind boundary (last known u). All targets are RELATIVE
    # to this, so the model predicts the structural DRIFT through the blind zone
    # (bounded, learnable) instead of an unknowable per-well absolute offset.
    if known.sum() >= 1:
        u_known = (tvt_in + z)
        u_last = float(u_known[known][-1])
    else:
        u_last = 0.0
    # field expressed as increment from its own last-known value (same relative frame)
    if known.sum() >= 20:
        fe_last = float(np.median(fe[known][-20:]))
    elif known.sum() >= 1:
        fe_last = float(fe[known][-1])
    else:
        fe_last = 0.0
    field_delta = fe - fe_last
    known_u_rel = np.where(known, (tvt_in + z) - u_last, 0.0)   # known drift (0 at boundary)
    md_rel = (md - md[known][-1]) / 1000.0 if known.sum() >= 1 else (md - md[0]) / 1000.0
    zc = z - np.polyval(np.polyfit(md, z, 1), md)

    ch = np.stack([
        (grf - med) / 30.0,
        (grs - med) / 30.0,
        dgr / 5.0,
        field_delta / 50.0,                 # field DRIFT (not absolute)
        np.clip(db / CFG['sup_cut'], 0, 2.0),
        known.astype(np.float64),
        known_u_rel / 50.0,                 # known-zone drift, boundary-relative
        zc / 50.0,
        md_rel,                             # ft from boundary / 1000
    ], axis=1).astype(np.float32)           # [n, 9]

    if is_test:
        target = np.zeros(n, np.float32)
        tmask = blind.astype(np.float32)
    else:
        tvt = h['TVT'].values.astype(np.float64)
        target = ((tvt + z) - u_last).astype(np.float32)        # INCREMENT target
        tmask = (blind & np.isfinite(tvt)).astype(np.float32)   # G3: blind only
    return ch, target, tmask, z, blind, u_last


In [6]:
# ==============================================================================
# CELL 6 — Assemble the training set as padded tensors
# ==============================================================================
def datum_for(h, is_test=False):
    """Deployment-safe datum: median of KNOWN-zone u (needs no TVT/Geology)."""
    z = zfix(h)
    tvt_in = h['TVT_input'].values.astype(np.float64)
    known = np.isfinite(tvt_in)
    if known.sum() >= 20:
        return float(np.median((tvt_in + z)[known]))
    return 0.0

def assemble(wells, split_dir, tw_dim=len(TW_GRID)):
    seqs, tgts, masks, tws, lens, metas = [], [], [], [], [], []
    for w in wells:
        h, t = load_well(split_dir, w)
        d = datum_for(h)
        ch, tg, tm, z, blind, u_last = build_sequence(h, t, w, d, is_test=(split_dir==TEST_DIR))
        if tm.sum() < 50 and split_dir != TEST_DIR:
            continue
        L = min(len(ch), CFG['win_max'])
        seqs.append(ch[:L]); tgts.append(tg[:L]); masks.append(tm[:L])
        tws.append(typewell_context(t, d)); lens.append(L)
        metas.append(dict(well=w, z=z[:L], blind=blind[:L], datum=d, u_last=u_last, n_full=len(ch)))
    return seqs, tgts, masks, tws, lens, metas

print('assembling training sequences (leak-free)...')
t0 = time.time()
SEQS, TGTS, MASKS, TWS, LENS, METAS = assemble(train_wells, TRAIN_DIR)
print(f'{len(SEQS)} usable wells ({time.time()-t0:.0f}s) | '
      f'median len {int(np.median(LENS))}, max {max(LENS)}')
N_CH = SEQS[0].shape[1]; TW_DIM = len(TWS[0])
print('channels:', N_CH, '| typewell dim:', TW_DIM)


assembling training sequences (leak-free)...
773 usable wells (324s) | median len 6576, max 12141
channels: 9 | typewell dim: 64


In [7]:
# ===== CELL 6.5 — anchor band: supervise the handoff (lever 1) =====
BAND = int(CFG['anchor_band'])
MASKS_TRAIN, K0S = [], []
n_band = 0
for i in range(len(SEQS)):
    b = METAS[i]['blind']
    k0 = int(np.argmax(b)) if b.any() else len(b)     # first blind station
    K0S.append(k0)
    m = MASKS[i].copy()
    lo = max(0, k0 - BAND)
    if k0 > lo:
        band = np.zeros_like(m)
        band[lo:k0] = 1.0
        band *= np.isfinite(TGTS[i]).astype(np.float32)
        m = np.maximum(m, band)
        n_band += int(band.sum())
    MASKS_TRAIN.append(m)

print('anchor band: +%d stations (%.1f%% of the train loss mask)'
      % (n_band, 100.0 * n_band / sum(float(x.sum()) for x in MASKS_TRAIN)))
print('SANITY target at boundary-1: mean |t| = %.5f  (must be ~0)'
      % np.mean([abs(float(TGTS[i][K0S[i]-1])) for i in range(len(SEQS)) if K0S[i] > 0]))

anchor band: +154600 stations (3.9% of the train loss mask)
SANITY target at boundary-1: mean |t| = 0.00000  (must be ~0)


In [8]:
# ==============================================================================
# CELL 7 — Normalization stats on TRAIN ONLY (G4) — refit inside each fold
# ==============================================================================
def fit_norm(seqs, idxs):
    allc = np.concatenate([seqs[i] for i in idxs], axis=0)
    mu = allc.mean(0); sd = allc.std(0) + 1e-6
    return mu.astype(np.float32), sd.astype(np.float32)

In [9]:
# ==============================================================================
# CELL 8 — 1D U-Net model
# ==============================================================================
class ConvBlock(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(ci, co, 5, padding=2), nn.BatchNorm1d(co), nn.GELU(),
            nn.Conv1d(co, co, 5, padding=2), nn.BatchNorm1d(co), nn.GELU())
    def forward(self, x): return self.net(x)

class UNet1D(nn.Module):
    def __init__(self, n_ch, tw_dim, base=96):
        super().__init__()
        self.tw = nn.Sequential(nn.Linear(tw_dim, 64), nn.GELU(), nn.Linear(64, base))
        self.inc = ConvBlock(n_ch + base, base)
        self.d1 = ConvBlock(base, base*2); self.d2 = ConvBlock(base*2, base*4)
        self.d3 = ConvBlock(base*4, base*4)
        self.pool = nn.MaxPool1d(2)
        self.up = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)
        self.u2 = ConvBlock(base*4 + base*4, base*2)
        self.u1 = ConvBlock(base*2 + base*2, base)
        self.u0 = ConvBlock(base + base, base)
        self.outc = nn.Conv1d(base, 1, 1)
    def forward(self, x, tw):
        # x: [B, n_ch, L]  tw: [B, tw_dim]
        B, _, L = x.shape
        g = self.tw(tw)[:, :, None].expand(-1, -1, L)   # broadcast typewell context
        x = torch.cat([x, g], dim=1)
        x0 = self.inc(x)
        x1 = self.d1(self.pool(x0)); x2 = self.d2(self.pool(x1))
        x3 = self.d3(self.pool(x2))
        y = self.u2(torch.cat([self._match(self.up(x3), x2), x2], 1))
        y = self.u1(torch.cat([self._match(self.up(y), x1), x1], 1))
        y = self.u0(torch.cat([self._match(self.up(y), x0), x0], 1))
        return self.outc(y)[:, 0, :]                      # [B, L] predicted u
    @staticmethod
    def _match(a, ref):
        if a.shape[-1] != ref.shape[-1]:
            a = F.interpolate(a, size=ref.shape[-1], mode='linear', align_corners=False)
        return a

class BiGRUNet(nn.Module):
    def __init__(self, n_ch, tw_dim, base=96):
        super().__init__()
        self.tw = nn.Sequential(nn.Linear(tw_dim, 64), nn.GELU(), nn.Linear(64, 48))
        self.proj = nn.Conv1d(n_ch + 48, 96, 5, padding=2)
        self.gru = nn.GRU(96, base, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.1)
        self.head = nn.Sequential(nn.Linear(2 * base, 96), nn.GELU(), nn.Linear(96, 1))
 
    def forward(self, x, tw, lens=None):
        B, _, L = x.shape
        g = self.tw(tw)[:, :, None].expand(-1, -1, L)
        x = torch.cat([x, g], dim=1)
        z = torch.relu(self.proj(x)).transpose(1, 2)        # [B, L, 96]
        if lens is not None:
            pk = nn.utils.rnn.pack_padded_sequence(
                z, lens.cpu(), batch_first=True, enforce_sorted=False)
            o, _ = self.gru(pk)
            o, _ = nn.utils.rnn.pad_packed_sequence(
                o, batch_first=True, total_length=L)
        else:
            o, _ = self.gru(z)
        return self.head(o)[:, :, 0]


In [10]:
# ==============================================================================
# CELL 9 — Batching (pad wells to equal length within a batch)
# ==============================================================================
def make_batch(idxs, seqs, tgts, masks, tws, mu, sd, augment=False):
    L = max(len(seqs[i]) for i in idxs)
    B = len(idxs)
    X = np.zeros((B, N_CH, L), np.float32)
    Y = np.zeros((B, L), np.float32)
    M = np.zeros((B, L), np.float32)
    T = np.zeros((B, TW_DIM), np.float32)
    Ls = np.zeros(B, np.int64)                                  # NEW
    for b, i in enumerate(idxs):
        c = (seqs[i] - mu) / sd
        if augment:
            g = 1.0 + np.random.uniform(-CFG['aug_gain'], CFG['aug_gain'])
            o = np.random.uniform(-CFG['aug_offset'], CFG['aug_offset']) / 30.0
            c[:, 0] = c[:, 0] * g + o; c[:, 1] = c[:, 1] * g + o
        n = len(c)
        X[b, :, :n] = c.T; Y[b, :n] = tgts[i]; M[b, :n] = masks[i]; T[b] = tws[i]
        Ls[b] = n                                               # NEW
    return (torch.tensor(X, device=DEVICE), torch.tensor(T, device=DEVICE),
            torch.tensor(Y, device=DEVICE), torch.tensor(M, device=DEVICE),
            torch.tensor(Ls))                                   # NEW — stays on CPU deliberately

def masked_l1(pred, y, m):
    d = (pred - y).abs() * m
    return d.sum() / m.sum().clamp(min=1.0)

In [ ]:
# ===== CELL 10 — training: packed GRU + anchor band + EMA =====
def evaluate_model(mdl, va, mu, sd, recenter=False):
    """Blind-only MAE (competition metric). recenter=True also reports the
    boundary-offset-corrected variant as a diagnostic."""
    mdl.eval(); raw, rec = [], []
    with torch.no_grad():
        for i in va:
            X, T, Y, M, Ls = make_batch([i], SEQS, TGTS, MASKS, TWS, mu, sd)
            p = mdl(X, T, Ls)
            raw.append((((p - Y).abs() * M).sum() / M.sum().clamp(min=1)).item())
            if recenter:
                k0 = K0S[i]; lo = max(0, k0 - int(CFG['anchor_band']))
                if k0 > lo:
                    off = (p[0, lo:k0] - Y[0, lo:k0]).median()
                    rec.append(((((p - off) - Y).abs() * M).sum()
                                / M.sum().clamp(min=1)).item())
                else:
                    rec.append(raw[-1])
    return float(np.mean(raw)), (float(np.mean(rec)) if recenter else None)


def train_folds():
    np.random.seed(0); perm = np.random.permutation(np.arange(len(SEQS)))
    folds = np.array_split(perm, CFG['n_folds'])
    oof_mae = np.full(len(SEQS), np.nan); fold_models = []
    d = CFG['ema_decay']
    for k in range(CFG['n_folds']):
        va = folds[k]
        tr = np.concatenate([folds[j] for j in range(CFG['n_folds']) if j != k])
        mu, sd = fit_norm(SEQS, tr)
        model = BiGRUNet(N_CH, TW_DIM, CFG['hidden']).to(DEVICE)
        ema_model = BiGRUNet(N_CH, TW_DIM, CFG['hidden']).to(DEVICE)
        ema = {kk: v.detach().clone() for kk, v in model.state_dict().items()}
        opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'],
                                weight_decay=CFG['weight_decay'])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG['epochs'])
        best_raw = best_ema = best_all = 1e9
        best_state, best_tag, bad = None, 'raw', 0

        for ep in range(CFG['epochs']):
            model.train(); np.random.shuffle(tr)
            for s in range(0, len(tr), CFG['batch_wells']):
                bi = tr[s:s+CFG['batch_wells']]
                # NOTE: MASKS_TRAIN here (blind + anchor band)
                X, T, Y, M, Ls = make_batch(bi, SEQS, TGTS, MASKS_TRAIN, TWS,
                                            mu, sd, augment=True)
                opt.zero_grad()
                loss = masked_l1(model(X, T, Ls), Y, M)
                loss.backward(); opt.step()
                with torch.no_grad():                       # per-step EMA
                    for kk, v in model.state_dict().items():
                        if v.dtype.is_floating_point:
                            ema[kk].mul_(d).add_(v.detach(), alpha=1.0 - d)
                        else:
                            ema[kk].copy_(v)
            sched.step()

            v_raw, v_rec = evaluate_model(model, va, mu, sd, recenter=True)
            improved = False
            if v_raw < best_raw - 1e-4:
                best_raw = v_raw; improved = True
            if v_raw < best_all - 1e-4:
                best_all = v_raw; best_tag = 'raw'
                best_state = {kk: vv.cpu().clone() for kk, vv in model.state_dict().items()}

            v_ema = None
            if ep >= CFG['ema_warmup'] and ep % CFG['ema_eval_every'] == 0:
                ema_model.load_state_dict({kk: vv.to(DEVICE) for kk, vv in ema.items()})
                v_ema, _ = evaluate_model(ema_model, va, mu, sd)
                if v_ema < best_ema - 1e-4:
                    best_ema = v_ema; improved = True
                if v_ema < best_all - 1e-4:
                    best_all = v_ema; best_tag = 'ema'
                    best_state = {kk: vv.cpu().clone() for kk, vv in ema.items()}

            bad = 0 if improved else bad + 1
            if ep % 10 == 0 or bad >= CFG['patience']:
                print('  fold %d ep %3d: raw %.3f (best %.3f) | recentered %.3f | ema %s'
                      % (k, ep, v_raw, best_raw, v_rec,
                         ('%.3f' % v_ema) if v_ema is not None else '-'), flush=True)
            if bad >= CFG['patience']:
                break

        model.load_state_dict(best_state)
        fold_models.append((model, mu, sd))
        for i in va: oof_mae[i] = best_all
        print('FOLD %d: raw %.3f | ema %.3f -> shipping %s (%.3f)'
              % (k, best_raw, best_ema, best_tag, best_all))
    return fold_models, oof_mae


print('training (GPU)...')
FOLD_MODELS, OOF = train_folds()
print('\nOOF blind-MAE (mean over wells): %.3f' % np.nanmean(OOF))
print('NOTE: full wells now (win_max=%d) — this scores the FAR TOE for the first time.'
      % CFG['win_max'])

training (GPU)...
  fold 0 ep   0: raw 42.898 (best 42.898) | recentered 45.852 | ema -
  fold 0 ep  10: raw 18.939 (best 18.939) | recentered 19.247 | ema -
  fold 0 ep  20: raw 17.139 (best 16.786) | recentered 18.965 | ema 40.674
  fold 0 ep  30: raw 25.595 (best 15.126) | recentered 30.195 | ema 26.972
  fold 0 ep  40: raw 15.716 (best 15.126) | recentered 17.902 | ema 32.182
  fold 0 ep  50: raw 14.365 (best 14.213) | recentered 15.567 | ema 21.099
  fold 0 ep  60: raw 14.687 (best 13.516) | recentered 16.150 | ema 15.147
  fold 0 ep  70: raw 15.715 (best 13.499) | recentered 16.668 | ema 13.754
  fold 0 ep  80: raw 14.422 (best 13.499) | recentered 15.571 | ema 13.535
  fold 0 ep  90: raw 13.736 (best 13.321) | recentered 14.901 | ema 13.449
  fold 0 ep 100: raw 13.705 (best 13.258) | recentered 14.738 | ema 13.408
  fold 0 ep 110: raw 14.020 (best 13.258) | recentered 15.044 | ema 13.445
  fold 0 ep 120: raw 13.573 (best 13.258) | recentered 14.447 | ema 13.449
  fold 0 ep 130: 

In [ ]:
# ==============================================================================
# CELL 11 — Blend evaluation: does model + field beat field alone, out-of-well?
# ==============================================================================
# The safe deployment is an ENSEMBLE: blend model u-hat with the field u by
# inverse-variance-style confidence. Here we just measure whether the model's OOF
# prediction, blended, beats the field baseline — a second honest gate.
# (Full blend with v22 happens in your classical notebook; see Step 8 of the guide.)

In [ ]:
# ==============================================================================
# CELL 12 — Inference on the hidden test set + write submission.csv
# ==============================================================================
def predict_test():
    seqs, tgts, masks, tws, lens, metas = assemble(test_wells, TEST_DIR)
    rows = []
    for j, meta in enumerate(metas):
        w = meta['well']; z = meta['z']; blind = meta['blind']
        preds = []
        for (model, mu, sd) in FOLD_MODELS:       # ensemble the 5 folds
            model.eval()
            with torch.no_grad():
                # X, T, Y, M = make_batch([j], seqs, tgts, masks, tws, mu, sd)
                # p = model(X, T)[0, :len(z)].cpu().numpy()
                X, T, Y, M, Ls = make_batch([j], seqs, tgts, masks, tws, mu, sd)
                p = model(X, T, Ls)[0, :len(z)].cpu().numpy()
            preds.append(p)
        u_hat = np.mean(preds, axis=0) + metas[j]['u_last']  # undo increment: u = delta + u_last
        tvt_hat = u_hat - z                                   # u = TVT + Z => TVT = u - Z
        # Emit ONLY blind-zone rows, keyed by ABSOLUTE station index (id = well_rowidx).
        # (sample_submission contains exactly the blind rows.)
        h, _ = load_well(TEST_DIR, w)
        blindfull = ~np.isfinite(h['TVT_input'].values.astype(np.float64))
        for k in np.where(blindfull)[0]:
            if k < len(tvt_hat):
                rows.append((f'{w}_{k}', float(tvt_hat[k])))
            else:
                # station beyond win_max truncation: fall back to last predicted value
                rows.append((f'{w}_{k}', float(tvt_hat[-1])))
    sub = pd.DataFrame(rows, columns=['id', 'tvt'])
    return sub

print('predicting test + writing submission...')
sub = predict_test()
# align to sample_submission id order
ss = pd.read_csv(os.path.join(ROOT, 'sample_submission.csv'))
sub = ss[['id']].merge(sub, on='id', how='left')
assert sub['tvt'].notna().all(), 'missing predictions — check id alignment'
sub.to_csv('submission.csv', index=False)
print('wrote submission.csv:', sub.shape)
print(sub.head())


In [ ]:
# ==============================================================================
# CELL 13 — Save model weights (so you can blend with the classical notebook)
# ==============================================================================
torch.save({'folds': [(m.state_dict(), mu, sd) for (m, mu, sd) in FOLD_MODELS],
            'cfg': CFG, 'tw_grid': TW_GRID}, 'seq_model.pt')
np.save('oof_mae.npy', OOF)
print('saved seq_model.pt and oof_mae.npy — download these from the Output tab.')